<a href="https://colab.research.google.com/github/Okahak71/learning_ai/blob/main/makemore/MLP/makemore_MLP_v3(initialization_fix).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Trying to beat Andrej's loss function from MLP video
import time
import torch as t
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

t.set_default_device('cuda')

In [2]:
!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

--2026-09-28 13:53:23--  https://raw.githubusercontent.com/karpathy/makemore/master/names.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228145 (223K) [text/plain]
Saving to: ‘names.txt’

names.txt           100%[===================>] 222.80K  --.-KB/s    in 0.02s   

2026-09-28 13:53:24 (13.3 MB/s) - ‘names.txt’ saved [228145/228145]



In [3]:
print_every = 5000
start_time = time.time()
last_time = time.time()

words = open("names.txt", 'r').read().splitlines()
st = sorted(list(set(''.join(words))))
stoi = {s:i + 1 for i, s in enumerate(st)}
stoi["."] = 0
itos = {i : s for s, i in stoi.items()}

#Hyperparameters
learning_rate = 0.1
vocab_size = 27
block_size = 5
neurons = 200
dim = 10

#-------------
def build_dataset(words):
    xs, ys = [], []
    for w in words:
        context = [0] * block_size
        for c in w + '.':
            xs.append(context)
            ys.append(stoi[c])
            context = context[1:] + [stoi[c]]
    xs = t.tensor(xs)
    ys = t.tensor(ys)
    return xs, ys

#Training - 80%, Val/Dev - 10%, Testing - 10%
import random
random.seed(42)
random.shuffle(words)
sz1 = int(0.8 * len(words))
sz2 = int(0.9 * len(words))
x_tr, y_tr = build_dataset(words[:sz1])
x_dev, y_dev = build_dataset(words[sz1:sz2])
x_test, y_test = build_dataset(words[sz2:])

#-----------Parameters---------
rng = t.Generator(device='cuda').manual_seed(2147483647)
C = t.randn((vocab_size, dim), generator=rng)  # THE Lookup Table

#first layer
w1 = t.randn((block_size * dim, neurons), generator=rng) / (block_size*dim)**0.5   # THE FIXES #|
b1 = t.randn((neurons), generator=rng) * 0.01                                                  #|

#second layer
w2 = t.randn((neurons, vocab_size), generator=rng) * 0.01                                      #|
b2 = t.randn(vocab_size, generator=rng) * 0.01                                                 #|

parameters = [C, w1, b1, w2, b2]

for p in parameters:
    p.requires_grad = True
total = 0

In [4]:
ls = []

for i in range(120000):
    total += 1
    # Minibatch construction
    id = t.randint(0, x_tr.shape[0], (256,))

    #forward pass
    emb = C[x_tr[id]]
    h = t.tanh(emb.view(-1, block_size * dim) @ w1 + b1)
    logits = h @ w2 + b2
    loss = F.cross_entropy(logits, y_tr[id])

    ls.append(loss.log10().item())

    # print(loss.item())
    if i < 2:
        print(f"Initial loss = {loss.item()}")

    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    #update
    lr = 0.1 if i < 100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    #------------------------------
    if i > 0 and i % print_every == 0:
        current_time = time.time()
        elapsed_chunk = current_time - last_time
        iterations_per_second = print_every / elapsed_chunk
        print(f"\rIteration: {i} | Speed: {iterations_per_second:.2f} it/s | loss = {loss}")
        last_time = current_time

Initial loss = 3.308901309967041
Initial loss = 3.2571139335632324
Iteration: 5000 | Speed: 222.44 it/s | loss = 2.2150423526763916
Iteration: 10000 | Speed: 560.77 it/s | loss = 2.1475634574890137
Iteration: 15000 | Speed: 719.68 it/s | loss = 2.110503911972046
Iteration: 20000 | Speed: 691.07 it/s | loss = 2.1253623962402344
Iteration: 25000 | Speed: 685.37 it/s | loss = 2.067471504211426
Iteration: 30000 | Speed: 704.20 it/s | loss = 1.9114398956298828
Iteration: 35000 | Speed: 541.81 it/s | loss = 2.006828546524048
Iteration: 40000 | Speed: 553.05 it/s | loss = 1.8900727033615112
Iteration: 45000 | Speed: 675.40 it/s | loss = 2.115741729736328
Iteration: 50000 | Speed: 641.28 it/s | loss = 1.987969160079956
Iteration: 55000 | Speed: 608.36 it/s | loss = 1.9975738525390625
Iteration: 60000 | Speed: 627.90 it/s | loss = 1.9453330039978027
Iteration: 65000 | Speed: 639.27 it/s | loss = 1.8918735980987549
Iteration: 70000 | Speed: 716.57 it/s | loss = 1.8524982929229736
Iteration: 7500

In [5]:
# @t.no_grad()
emb = C[x_dev]
h = t.tanh(emb.view(-1, dim * block_size) @ w1 + b1)
logits = h @ w2 + b2
loss = F.cross_entropy(logits, y_dev)
print(loss.item())

2.0499279499053955
current learning_rate : 0.01
120000


In [7]:
emb = C[x_test]
h = t.tanh(emb.view(-1, dim * block_size) @ w1 + b1)
logits = h @ w2 + b2
loss = F.cross_entropy(logits, y_test)
print(f"FINAL LOSS TEST : {loss.item()}")

FINAL LOSS TEST : 2.049778699874878
